# Chapitre 6 — Évaluer un modèle et mesurer l'incertitude · Corrigé commenté

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch06_corrige.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Chaque réponse commence par le raisonnement sur papier, puis montre son équivalent en Python. Les commentaires signalent, à chaque calcul, le lot utilisé.

**Sommaire.** Données · Les calculs des leçons 6.1 à 6.6 · Exercices 6.1 à 6.6

**Le protocole, dans tout le notebook :** ajuster sur l'**entraînement**, choisir sur la **validation**, évaluer **une seule fois** sur le **test**. Pour un test A/B : comparer deux groupes **tirés au sort**, et donner l'écart **avec son intervalle de confiance**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur par lot
SET_TRAIN, SET_VAL, SET_TEST = '#2e7d32', '#b84d00', '#1565c0'   # entraînement (vert), validation (orange foncé), test (bleu)
SPAM, LEGIT = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, RESIDUAL, THRESHOLD = '#757575', '#00897b', '#c2185b', '#8e24aa'   # test A/B : A gris, B sarcelle


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def mean_loss(labels, X, w, b):
    """Perte logarithmique moyenne (équations 5.6 et 5.7)."""
    p = np.clip(sigmoid(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(labels, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


FEATURE_NAMES = ['links', 'suspicious_words', 'length', 'capitals', 'hour', 'attachments', 'size', 'recipients']


def make_messages(seed=77):
    """Messages fictifs (exercice 6.4 et leçon 6.4) : 40 groupes, campagnes de spams (groupes pairs) ou
    expéditeurs légitimes (groupes impairs). Les messages d'un groupe sont presque identiques. Seules les deux
    premières caractéristiques dépendent de l'étiquette ; les huit sont dans des unités très différentes."""
    rng = np.random.default_rng(seed)
    units = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    offsets = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    rows = []
    for g in range(40):
        spam = g % 2 == 0
        size = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(size):
            rows.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * units + offsets))
    return pd.DataFrame(rows, columns=['group', 'spam'] + FEATURE_NAMES)


def fit(X, y, lam, l1=False, step_k=0.1, n_steps=3000):
    """Descente de gradient sur la perte pénalisée : L2 (équation 6.2) ou L1 (équation 6.3). b n'est pas pénalisé."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_steps):
        gap = sigmoid(X @ w + b) - y
        gw, gb = X.T @ gap / len(y), gap.mean()
        if l1:   # pas de gradient, puis on rapproche chaque poids de 0 d'au plus pas × λ (sans le faire changer de signe)
            w = w - step_k * gw
            w = np.sign(w) * np.maximum(np.abs(w) - step_k * lam, 0)
        else:    # la dérivée de λ × somme des w² est 2λw
            w = w - step_k * (gw + 2 * lam * w)
        b -= step_k * gb
    return w, b
ajuster = fit   # alias du livre (nom de l'édition française imprimée)

## Données

Les identifiants D6.1 à D6.5, D6.E1, D6.E2, D6.E3, D6.E5 et D5.1 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 6, recopiées des fichiers CSV du livre (ne pas modifier)

# D6.1 — les trois lots de l'association : messages, spams, et erreurs du filtre pour un seuil
subsets = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.2 — deux découpages des mêmes messages : le lot de test, au seuil 0,7
leak = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['split', 'messages', 'spams', 'copies_seen', 'missed_copies', 'missed_new', 'fp'])

# D6.3 — six messages nouveaux (validation de la leçon 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D5.1 — les douze messages du chapitre 5 (entraînement de la leçon 6.3)
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D6.E2 — la médiathèque (exercice 6.2) : lots de validation et de test
media_library = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.4 — les trois plis de l'association (leçon 6.4) : liens (moyenne, écart-type) et pertes de validation
folds = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['fold', 'messages', 'spams', 'mean_links', 'std_links', 'loss_l001', 'loss_l01'])

# D6.5 — le test A/B de la librairie (leçons 6.5 et 6.6)
bookshop = pd.DataFrame([
    ('A', 'lettre actuelle', 2500, 200),
    ('B', 'livraison offerte', 2400, 240)
], columns=['version', 'description', 'customers', 'orders'])

# D6.E1 — validation croisée de la médiathèque (exercice 6.1) : perte de chaque pli pour cinq λ
folds_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['fold', 'lambda', 'loss'])

# D6.E3 — répartition de quatre groupes dans les plis du bénévole (exercice 6.3)
folds_of_groups = pd.DataFrame([
    ('Montres', 'campagne de spams', 9, 8, 9),
    ('Colis en attente', 'campagne de spams', 5, 6, 4),
    ('Loterie', 'campagne de spams', 7, 7, 8),
    ('Lettre de la mairie', 'expéditeur légitime', 4, 3, 5)
], columns=['group', 'kind', 'fold_1', 'fold_2', 'fold_3'])

# D6.E5 — le test A/B du théâtre (exercice 6.5)
theater = pd.DataFrame([
    ('A', 'lettre habituelle', 1800, 126),
    ('B', 'lettre avec vidéo', 1750, 147)
], columns=['version', 'description', 'audience', 'bookings'])

### Les variables des données

Les noms de code sont en anglais. Pour les colonnes, la dernière colonne donne le nom utilisé dans les fichiers CSV du livre.

| Nom | Sens | Unité | Colonne du CSV |
|:--|:--|:--|:--|
| `message` | identifiant du message (M1…, R1…) | — | — |
| `label` | étiquette réelle : spam ou légitime | — | `etiquette` |
| `links` | nombre de liens dans le message | liens | `liens` |
| `suspicious_words` | nombre de mots suspects dans le message | mots | `mots_suspects` |
| `messages` | nombre de messages | messages | — |
| `spams` | nombre de spams | messages | — |
| `subset` | lot : entraînement, validation ou test | — | `lot` |
| `threshold` | seuil de décision | — | `seuil` |
| `campaign` | campagne d'envoi du message | — | `campagne` |
| `group` | groupe de messages (une campagne entière ou un expéditeur), à garder entier dans un pli | — | `groupe` |
| `kind` | nature du groupe : campagne de spams ou expéditeur légitime | — | `nature` |
| `fold` | numéro du pli de validation croisée | — | `pli` |
| `fold_1` | D6.E3 : nombre de messages du groupe dans le pli 1 | messages | `pli_1` |
| `fold_2` | D6.E3 : nombre de messages du groupe dans le pli 2 | messages | `pli_2` |
| `fold_3` | D6.E3 : nombre de messages du groupe dans le pli 3 | messages | `pli_3` |
| `mean_links` | D6.4 : moyenne des liens sur les messages d'entraînement du tour (les deux autres plis) | liens | `moyenne_liens` |
| `std_links` | D6.4 : écart-type des liens sur les messages d'entraînement du tour (les deux autres plis) | liens | `ecart_type_liens` |
| `description` | description de la version testée | — | — |
| `audience` | nombre de spectateurs | personnes | `spectateurs` |
| `bookings` | nombre de réservations | réservations | `reservations` |
| `customers` | nombre de clients exposés à la version | clients | `clients` |
| `orders` | nombre de commandes | commandes | `commandes` |
| `version` | version de la campagne (A ou B) | — | — |
| `twelve` | les douze messages du chapitre 5 | — | — |
| `loss_l001` | D6.4 : perte de validation du pli, avec λ = 0,01 | — | `perte_l001` |
| `loss_l01` | D6.4 : perte de validation du pli, avec λ = 0,1 | — | `perte_l01` |
| `fn` | nombre de faux négatifs (spams manqués) | messages | — |
| `fp` | nombre de faux positifs (messages légitimes signalés) | messages | — |
| `split` | D6.2 : découpage du lot de test (au hasard ou par campagne) | — | `decoupage` |
| `copies_seen` | D6.2 : spams du test copiés d'une campagne vue à l'entraînement | messages | `copies_vues` |
| `missed_copies` | D6.2 : copies manquées par le filtre | messages | `manques_copies` |
| `missed_new` | D6.2 : spams nouveaux manqués par le filtre | messages | `manques_nouveaux` |
| `lambda` | D6.E1 : force λ de la pénalité | — | — |
| `loss` | D6.E1 : perte de validation du pli pour ce λ | — | `perte` |

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 6.1 à 6.6.

In [ ]:
# Leçon 6.1 — coût par message (équation 6.1), k = 3 pour l'association
k = 3
t = subsets.copy()
t['coût'] = t['fn'] + k * t['fp']
t['coût par message'] = (t['coût'] / t['messages']).round(3)    # on divise par la taille DU LOT
print(t.to_string(index=False))
val = t[t['subset'] == 'validation']
chosen = val.loc[val['coût par message'].idxmin(), 'threshold']
print('seuil choisi sur la validation :', chosen)
test = t[(t['subset'] == 'test') & (t['threshold'] == chosen)].iloc[0]
print(f"test, une seule fois : coût {test['coût']} sur {test['messages']} messages = {test['coût par message']:.3f} par message")

a7 = t[t['threshold'] == 0.7]
colors = {'entrainement': SET_TRAIN, 'validation': SET_VAL, 'test': SET_TEST}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['subset'], a7['coût par message'], color=[colors[l] for l in a7['subset']])
plt.ylabel('Coût par message (seuil 0,7)'); plt.show()

In [ ]:
# Leçon 6.2 — une fuite de données : rappel et coût par message des deux découpages
for _, l in leak.iterrows():
    missed = l['missed_copies'] + l['missed_new']
    new_spams = l['spams'] - l['copies_seen']
    print(f"{l['split']:>12} : rappel = {(l['spams'] - missed) / l['spams']:.2f}, "
          f"coût par message = {(missed + 3 * l['fp']) / l['messages']:.3f}, "
          f"part des spams nouveaux manqués = {l['missed_new']} / {new_spams} = {l['missed_new'] / new_spams:.2f}")

# Découper les campagnes, pas les messages : petite démonstration avec 40 campagnes de taille variable
rng = np.random.default_rng(6)
campaign = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # numéro de campagne de chaque spam
at_random = rng.random(campaign.size) < 0.25                              # 25 % des MESSAGES au test
test_campaigns = rng.choice(40, size=10, replace=False)                         # 25 % des CAMPAGNES au test
by_campaign = np.isin(campaign, test_campaigns)
for name, test_ in (('au hasard', at_random), ('par campagne', by_campaign)):
    shared = len(set(campaign[test_]) & set(campaign[~test_]))
    print(f'{name:>12} : {test_.sum()} spams au test, {shared} campagnes présentes à la fois au test et ailleurs')

In [ ]:
# Leçon 6.3 — surapprentissage : filtres A et B sur l'entraînement (douze messages) et la validation (six)
X12, y12 = twelve[['links', 'suspicious_words']].to_numpy(float), (twelve['label'] == 'spam').to_numpy(int)
X6, y6 = six[['links', 'suspicious_words']].to_numpy(float), (six['label'] == 'spam').to_numpy(int)
filters = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for name, (w, b) in filters.items():
    p6 = sigmoid(X6 @ np.array(w) + b)
    losses6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'filtre {name} : entraînement {mean_loss(y12, X12, w, b):.2f}, validation {mean_loss(y6, X6, w, b):.2f} '
          f'| pertes des six : {np.round(losses6, 2)}')

# Perte pénalisée L2 (équation 6.2), λ = 0,05 ; la constante b n'est pas pénalisée
lam = 0.05
for name, (w, b) in filters.items():
    penalty = lam * np.sum(np.square(w))
    print(f'filtre {name} : somme des carrés des poids = {np.sum(np.square(w)):.4f}, pénalité = {penalty:.4f}, '
          f'perte pénalisée = {mean_loss(y12, X12, w, b) + penalty:.2f}')

# La pénalité dépend des unités : liens comptés en dizaines, mêmes prédictions, autre pénalité
X12_tens = X12 / np.array([10, 1])
w_tens = (10, 0.5)
print('mêmes probabilités :', np.allclose(sigmoid(X12_tens @ np.array(w_tens) - 4), sigmoid(X12 @ np.array([1, 0.5]) - 4)),
      f'| pénalité en unités : {lam * (1 + 0.25):.4f} ; en dizaines : {lam * (100 + 0.25):.4f}')

In [ ]:
# Leçon 6.4 — standardisation dans chaque tour : moyenne et écart-type des liens sur les DEUX AUTRES plis
for k in range(3):
    others = folds[folds['fold'] != k + 1]
    n = others['messages'].sum()
    m = (others['messages'] * others['mean_links']).sum() / n
    # écart-type de la réunion de deux plis (division par n) : moyenne des carrés, moins le carré de la moyenne
    v = (others['messages'] * (others['std_links'] ** 2 + others['mean_links'] ** 2)).sum() / n - m ** 2
    print(f'tour {k + 1} : validation = pli {k + 1}, entraînement = {n} messages, moyenne {m:.2f}, écart-type {np.sqrt(v):.2f}')
print(f'message à 6 liens du pli 3, au tour 3 : (6 - 2,25) / 1,76 = {(6 - 2.25) / 1.76:.2f}')

# Score de validation croisée (équation 6.4) des deux valeurs de λ
for col, lam in (('loss_l001', 0.01), ('loss_l01', 0.1)):
    print(f'λ = {lam} : pertes des plis {list(folds[col])}, VC = {folds[col].mean():.3f}')

In [ ]:
# Leçon 6.4 (fin) — L1 et L2 sur des messages fictifs à huit caractéristiques (les mêmes qu'à l'exercice 6.4)
msgs = make_messages()
X8 = msgs[FEATURE_NAMES].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # ici, standardisation sur tous les messages : simple démonstration
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = fit(X8, y8, 0.1)
w_l1, _ = fit(X8, y8, 0.1, l1=True)
print('poids L2 :', np.round(w_l2, 2))
print('poids L1 :', np.round(w_l1, 2))
print(f'poids nuls : L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(les deux premières caractéristiques sont les seules liées à l étiquette)')

In [ ]:
# Leçons 6.5 et 6.6 — le test A/B de la librairie
a, b_ = bookshop.iloc[0], bookshop.iloc[1]
pA, pB = a['orders'] / a['customers'], b_['orders'] / b_['customers']
gap = pB - pA
print(f'taux A = {100 * pA:.1f} %, taux B = {100 * pB:.1f} %, écart = {100 * gap:.1f} points, '
      f'variation relative = {100 * gap / pA:.0f} %')
seA, seB = np.sqrt(pA * (1 - pA) / a['customers']), np.sqrt(pB * (1 - pB) / b_['customers'])
se = np.sqrt(seA ** 2 + seB ** 2)                          # équation 6.6 : on additionne les CARRÉS
low, high = gap - 1.96 * se, gap + 1.96 * se           # équation 6.7
print(f'ES de A = {100 * seA:.2f} point, ES de B = {100 * seB:.2f} point, ES de l écart = {100 * se:.2f} point')
print(f'intervalle de confiance à 95 % : de {100 * low:.1f} à {100 * high:.1f} points')
print(f'pour 40 000 abonnés : {40000 * gap:.0f} commandes de plus (de {40000 * low:.0f} à {40000 * high:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0 : aucun effet')
plt.axvline(1, color=THRESHOLD, linestyle='--', label='seuil de rentabilité : 1 point')
plt.plot([100 * low, 100 * high], [0, 0], color=RESIDUAL, linewidth=4, label='intervalle à 95 %')
plt.scatter([100 * gap], [0], marker='D', s=70, color=RESIDUAL, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Effet de la version B (points de pourcentage)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Leçon 6.6 — le coût par message du test (leçon 6.1) est une moyenne : même raisonnement
costs = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 spams manqués, 4 fausses alertes, 226 sans coût
se_c = costs.std() / np.sqrt(len(costs))                     # écart-type (division par n) / racine de n
print(f'coût par message = {costs.mean():.3f}, écart-type = {costs.std():.3f}, ES = {se_c:.3f}, '
      f'intervalle : de {costs.mean() - 1.96 * se_c:.2f} à {costs.mean() + 1.96 * se_c:.2f}')

## Exercice 6.1 — Lire un graphique : trois plis, cinq valeurs de λ

*Notions : validation croisée (leçon 6.4, équation 6.4), surapprentissage et pénalité (leçon 6.3).* Données D6.E1 : la perte de validation de chaque pli de la médiathèque, pour cinq valeurs de $\lambda$.

### a) à e) Corrigé commenté

**a)** Pli 2 pour $\lambda = 0{,}1$ : 0,34. Pli 3 pour $\lambda = 0{,}001$ : 0,61.

**b)** Les moyennes des trois plis valent environ 0,51, 0,43, 0,36, 0,37 et 0,53. La plus petite est celle de $\lambda = 0{,}1$ (0,363). Pris seul, le pli 1 aurait choisi $\lambda = 1$ (0,35), le pli 2 $\lambda = 0{,}1$ (0,34) et le pli 3 $\lambda = 1$ (0,37) : deux plis sur trois préfèrent une autre valeur que la moyenne.

**c)** Pour $\lambda = 0{,}001$, la pénalité ne freine presque rien : le filtre apprend par cœur les particularités des plis d'entraînement, comme le filtre B de la leçon 6.3, et se trompe avec assurance sur le pli de validation. Pour $\lambda = 10$, la pénalité écrase tous les poids vers 0 : le filtre donne presque la même probabilité à tous les messages et ne distingue plus les spams. Entre les deux, la perte passe par un creux.

**d)** Les moyennes de $\lambda = 0{,}1$ et $\lambda = 1$ diffèrent d'environ 0,003. Pour un même $\lambda$, les plis diffèrent beaucoup plus : de 0,34 à 0,39 pour $\lambda = 0{,}1$. Le choix entre ces deux valeurs n'est donc pas solide : elles sont, au vu des données, aussi bonnes l'une que l'autre. En revanche, écarter 0,001 et 10 est un choix solide.

**e)** Le 0,36 est le score de validation croisée qui a servi à **choisir** $\lambda$ : il flatte légèrement le réglage retenu. La médiathèque doit d'abord ajuster le modèle final sur tous ses plis avec le $\lambda$ choisi, arrêter son seuil, puis mesurer **une seule fois** la perte sur un lot de test resté à l'écart. C'est ce nombre-là qu'elle peut annoncer.

In [ ]:
means = folds_lambda.groupby('lambda')['loss'].mean()     # VC(λ) : moyenne des trois plis
lam_chosen = means.idxmin()

In [ ]:
# Vérification de a) et b)
if means is None or lam_chosen is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    print('moyennes :', means.round(3).to_dict(), '→',
          'correct' if np.allclose(means.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'à revoir')
    print('λ choisi :', lam_chosen, '→', 'correct' if lam_chosen == 0.1 else 'à revoir')
    print('choix de chaque pli seul :', folds_lambda.loc[folds_lambda.groupby('fold')['loss'].idxmin(), ['fold', 'lambda']].values.tolist())
    for k, (style, marker) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = folds_lambda[folds_lambda['fold'] == k + 1]
        plt.plot(d['lambda'], d['loss'], style, marker=marker, color=SET_VAL, label=f'pli {k + 1}')
    plt.plot(means.index, means.values, 'D-', color='black', linewidth=2.5, label='moyenne')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Perte de validation'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Exercice 6.2 — Choisir, puis évaluer un seuil : la médiathèque

*Notions : trois lots (leçon 6.1), coût par message (équation 6.1), coût d'un seuil (équation 5.9).* Pour la médiathèque, une fausse alerte coûte deux spams manqués : $k = 2$. Données D6.E2. Faites d'abord les questions a) à e) sur papier, **sans regarder les lignes du test avant la question b)**.

### a) à e) Corrigé commenté

**a)** Sur la **validation** (150 messages), avec $FN + 2 \times FP$ :

| Seuil | Coût | Coût par message |
|:--|--:|--:|
| 0,4 | $3 + 2 \times 18 = 39$ | $39 / 150 = 0{,}260$ |
| 0,5 | $6 + 2 \times 9 = 24$ | $24 / 150 = 0{,}160$ |
| 0,6 | $10 + 2 \times 5 = 20$ | $20 / 150 \approx 0{,}133$ |
| 0,7 | $15 + 2 \times 3 = 21$ | $21 / 150 = 0{,}140$ |

Le seuil **0,6** coûte le moins sur la validation. On le retient.

**b)** Sur le **test** (240 messages), au seuil 0,6 seulement : $17 + 2 \times 9 = 35$, puis $35 / 240 \approx 0{,}146$ par message. C'est le nombre à annoncer.

**c)** Les deux lots n'ont pas la même taille : 150 et 240 messages. Les totaux, 20 et 35, feraient croire que le filtre coûte 75 % de plus sur le test. Par message, l'écart n'est que de 0,133 à 0,146, soit environ 9 % de plus. Un total grandit avec le nombre de messages ; un coût par message, non.

**d)** Oui. Le seuil a été choisi parce qu'il réussissait le mieux sur ces 150 messages-là, avec leur part de chance. Sa performance de validation est donc légèrement flatteuse. Le test, qui n'a servi à rien décider, ne bénéficie pas de cette chance.

**e)** Première erreur : il **choisit le seuil sur le test**, qui devient une seconde validation. Seconde erreur : il annonce le **meilleur** des quatre coûts du test, 0,117, qui est forcément flatteur : c'est un minimum, pas une mesure. Il devrait annoncer 0,146 par message, au seuil 0,6 choisi sur la validation. S'il tient à comparer 0,6 et 0,7, il lui faut une validation plus grande ou une validation croisée, puis un nouveau lot de test. Remarquez d'ailleurs que, sur la validation, 0,6 et 0,7 ne diffèrent que d'une unité de coût sur 150 messages : le choix entre eux est fragile.

In [ ]:
def cost_per_message(fn, fp, n, k):
    return (fn + k * fp) / n                    # on divise par la taille du lot
cout_par_message = cost_per_message   # alias du livre (nom de l'édition française imprimée)


def threshold_choice(validation_table, k):
    """Renvoie le seuil dont le coût par message est le plus petit sur la validation."""
    costs = [cost_per_message(l.fn, l.fp, l.messages, k) for l in validation_table.itertuples()]
    return validation_table['threshold'].iloc[int(np.argmin(costs))]   # le test n'est pas regardé ici
choisir_seuil = threshold_choice   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de a) et b), puis le protocole
k_med = 2
valid = media_library[media_library['subset'] == 'validation']
test_med = media_library[media_library['subset'] == 'test']
if cost_per_message(3, 18, 150, 2) is None or threshold_choice(valid, k_med) is None:
    print('pas encore complété. Complétez les fonctions.')
else:
    cv = [cost_per_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('validation, coût par message :', np.round(cv, 3), '→',
          'correct' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'à revoir')
    s = threshold_choice(valid, k_med)
    print(f'seuil choisi sur la validation : {s} →', 'correct' if s == 0.6 else 'à revoir')
    l = test_med[test_med['threshold'] == s].iloc[0]                    # UNE seule ligne du test
    ct = cost_per_message(l.fn, l.fp, l.messages, k_med)
    print(f'test, au seuil {s} seulement : {ct:.3f} par message →', 'correct' if abs(ct - 35 / 240) < 1e-9 else 'à revoir')
    # e) Ce que fait le collègue : il regarde TOUTES les lignes du test et garde la meilleure
    ct_all = [cost_per_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['threshold'].iloc[int(np.argmin(ct_all))]
    print(f'annonce du collègue : seuil {s_coll}, {min(ct_all):.3f} par message (choisi sur le test : trop optimiste) ;',
          f'annonce honnête : seuil {s}, {ct:.3f} par message →',
          'correct' if s_coll == 0.7 and abs(min(ct_all) - 28 / 240) < 1e-9 else 'à revoir')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['threshold'], cv, 'o-', color=SET_VAL, label='validation (sert à choisir)')
    plt.plot(test_med['threshold'], ct_all, 's--', color=SET_TEST, alpha=0.5, label='test (à ne pas regarder pour choisir)')
    plt.scatter([s], [ct], s=120, color=SET_TEST, zorder=3, label='test, au seuil choisi')
    plt.xlabel('Seuil'); plt.ylabel('Coût par message (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Corrigé commenté

La médiathèque peut annoncer : « Au seuil 0,6, choisi sur la validation, le filtre coûte environ 0,146 par message sur un lot de test de 240 messages qui n'avait servi à rien d'autre. » Le graphique montre pourquoi l'annonce du collègue est trop belle : en choisissant le point le plus bas de la courbe du test, il transforme le test en validation. Son 0,117 n'est plus une mesure du coût à venir. Enfin, 240 messages ne donnent pas un coût exact : la leçon 6.6 apprend à dire de combien il peut varier.

## Exercice 6.3 — Interpréter : une fuite entre les plis

*Notions : fuite de données (leçon 6.2), validation croisée par groupes et standardisation dans chaque tour (leçon 6.4).* Données D6.E3 : où sont tombés les messages de quatre groupes dans les plis du bénévole.

### a) à d) Corrigé commenté

**a)** Au tour 1, les plis 2 et 3 forment l'entraînement : le filtre a vu $8 + 9 = 17$ copies de la campagne « Montres ». Il en retrouve 9 dans le pli 1, en validation.

**b)** **Étape 2** : le pli est tiré message par message. Chaque campagne est donc coupée entre les plis (les quatre groupes du tableau le sont). À chaque tour, le filtre a vu à l'entraînement des copies presque identiques aux messages de validation : il reconnaît la campagne au lieu de juger un message nouveau. **Étape 1** : la moyenne et l'écart-type sont calculés sur les 800 messages, donc avec les messages de chaque pli de validation. Ces deux nombres font passer un peu d'information de la validation dans l'entraînement. Cette seconde fuite est plus petite, mais elle est du même genre.

**c)** La perte de test, 0,34, décrit le mieux les messages à venir : le lot de test a été formé par campagnes et n'a servi qu'une fois. Le score de validation croisée, 0,11, est trop beau parce que la validation contenait des copies déjà vues. Les étapes 4 et 5 sont correctes en elles-mêmes : choisir $\lambda$ sur la validation croisée, puis ouvrir le test une seule fois. Mais l'étape 4 travaille sur des scores faussés : la fuite récompense les filtres qui apprennent par cœur, si bien que le $\lambda$ retenu peut lui-même être trop petit.

**d)** Les groupes comptent 26, 15, 22 et 12 messages, soit 75 ; l'idéal serait 25 par pli. Une répartition possible : pli 1 = Montres (26), pli 2 = Loterie (22), pli 3 = Colis en attente et Lettre de la mairie (27). Étape 1 réécrite : « À chaque tour, calculer la moyenne et l'écart-type de chaque caractéristique sur les deux plis d'entraînement seulement, puis standardiser avec ces deux nombres les messages d'entraînement et ceux du pli de validation. »

In [ ]:
def split_groups(allocation):
    """allocation : un tableau avec une colonne 'group' et les colonnes 'fold_1', 'fold_2', 'fold_3'
    (nombre de messages du groupe dans chaque pli). Renvoie la liste des groupes coupés."""
    n_folds = (allocation[['fold_1', 'fold_2', 'fold_3']] > 0).sum(axis=1)   # dans combien de plis est le groupe ?
    return list(allocation.loc[n_folds > 1, 'group'])
groupes_coupes = split_groups   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de b) et d)
proposal = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['group', 'fold_1', 'fold_2', 'fold_3'])
if split_groups(folds_of_groups) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    print('plis du bénévole, groupes coupés :', split_groups(folds_of_groups), '→',
          'correct' if len(split_groups(folds_of_groups)) == 4 else 'à revoir')
    print('répartition corrigée, groupes coupés :', split_groups(proposal), '| tailles des plis :',
          proposal[['fold_1', 'fold_2', 'fold_3']].sum().tolist(), '→',
          'correct' if split_groups(proposal) == [] else 'à revoir')
    print('tour 1 : copies « Montres » vues à l entraînement =', int(folds_of_groups.loc[0, ['fold_2', 'fold_3']].sum()),
          ', en validation =', int(folds_of_groups.loc[0, 'fold_1']))

## Exercice 6.4 — Validation croisée par campagnes

*Notions : validation croisée (leçon 6.4, équation 6.4), pénalité L2 (équation 6.2), standardisation (équation 2.7), fuite (leçon 6.2).* Les messages sont fabriqués par `make_messages()` (cellule d'import). Les groupes 30 à 39 forment le lot de test : on n'y touche qu'à la question e).

In [ ]:
# Les messages de l'exercice : 30 groupes pour la validation croisée, 10 groupes mis de côté pour le test
msgs = make_messages()
test_ = msgs['group'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, FEATURE_NAMES].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'group'].to_numpy()
X_test, y_test = msgs.loc[test_, FEATURE_NAMES].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} messages, {msgs.group.nunique()} groupes ; validation croisée : {len(y_cv)} messages ; test : {len(y_test)} messages')
print(msgs[FEATURE_NAMES].describe().loc[['mean', 'std']].round(1))     # des unités très différentes

### a) Corrigé commenté

On s'attend à ce qu'elle **baisse puis remonte**. Sans pénalité, le filtre peut donner de grands poids aux six caractéristiques sans lien avec l'étiquette, qui varient par hasard d'un groupe à l'autre : il surapprend. Avec une pénalité trop forte, tous les poids sont écrasés.

In [ ]:
def group_folds(groups, K):
    """Numéro de pli (0 à K - 1) de chaque message : les groupes, rangés par ordre croissant, sont distribués
    à tour de rôle dans les plis ; tous les messages d'un groupe reçoivent le même pli."""
    ranks = {g: i % K for i, g in enumerate(np.unique(groups))}
    return np.array([ranks[g] for g in groups])
plis_par_groupe = group_folds   # alias du livre (nom de l'édition française imprimée)


def standardization(X_train, X_val):
    """Standardise X_train et X_val avec la moyenne et l'écart-type de X_train SEULEMENT."""
    m, s = X_train.mean(axis=0), X_train.std(axis=0)
    return (X_train - m) / s, (X_val - m) / s
standardiser = standardization   # alias du livre (nom de l'édition française imprimée)


def cv_score(lam, folds_, K=3):
    """Équation 6.4 : moyenne, sur les K tours, de la perte SANS pénalité sur le pli de validation."""
    losses = []
    for k in range(K):
        train, val = folds_ != k, folds_ == k
        Xe, Xv = standardization(X_cv[train], X_cv[val])          # nombres calculés dans le tour, sur l'entraînement
        w, b = fit(Xe, y_cv[train], lam)
        losses.append(mean_loss(y_cv[val], Xv, w, b))     # on juge les prédictions, sans pénalité
    return float(np.mean(losses))
score_vc = cv_score   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de b) et c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = group_folds(g_cv, 3)
if p_g is None or standardization(X_cv, X_cv)[0] is None or cv_score(0.1, p_g) is None:
    print('pas encore complété. Complétez les trois fonctions.')
else:
    splits = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'plis par groupe : {np.bincount(p_g).tolist()} messages, {splits} groupe(s) coupé(s) →',
          'correct' if splits == 0 and set(p_g) == {0, 1, 2} else 'à revoir')
    Xe, Xv = standardization(X_cv[p_g != 0], X_cv[p_g == 0])
    print('entraînement standardisé : moyennes ≈ 0 et écarts-types ≈ 1 ; validation : pas exactement →',
          'correct' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'à revoir')
    cv_group = {lam: cv_score(lam, p_g) for lam in LAMBDAS}
    lam_cv = min(cv_group, key=cv_group.get)
    print('VC par groupe :', {l: round(v, 3) for l, v in cv_group.items()}, '→ λ choisi :', lam_cv, '→',
          'correct' if lam_cv == 0.01 else 'à revoir')

In [ ]:
# d) Plis tirés message par message (à ne pas faire) : la fuite de la leçon 6.2, trois fois
if p_g is not None and cv_score(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    cv_message = {lam: cv_score(lam, p_m) for lam in LAMBDAS}
    print('VC message par message :', {l: round(v, 3) for l, v in cv_message.items()})
    plt.plot(range(5), list(cv_group.values()), 'o-', color=SET_VAL, label='plis par groupe')
    plt.plot(range(5), list(cv_message.values()), 's--', color='grey', label='plis message par message (fuite)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Score de validation croisée')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Modèle final : standardisation et poids sur les 30 groupes, avec le λ choisi ; test UNE seule fois
    Xc, Xt = standardization(X_cv, X_test)
    w_f, b_f = fit(Xc, y_cv, lam_cv)
    print(f'perte de test du modèle final (λ = {lam_cv}) : {mean_loss(y_test, Xt, w_f, b_f):.3f} ; '
          f'VC par groupe : {cv_group[lam_cv]:.3f} ; VC message par message : {cv_message[lam_cv]:.3f}')

### d) et e) Corrigé commenté

**c)** Avec des plis par groupe, le score de validation croisée vaut environ 0,547 pour $\lambda = 0$, 0,493 pour 0,001, **0,378 pour 0,01**, 0,424 pour 0,1 et 0,609 pour 1. Il baisse puis remonte : on retient $\lambda = 0{,}01$. Sans pénalité, le filtre surapprend sur les six caractéristiques sans lien avec l'étiquette ; avec $\lambda = 1$, les poids sont trop écrasés.

**d)** Avec des plis tirés message par message, les scores tombent à 0,207, 0,206, 0,246, 0,385 et 0,598. Chaque groupe est coupé entre les plis : à chaque tour, la validation contient des copies presque identiques de messages d'entraînement, et le filtre les reconnaît. Le score est trop beau, surtout pour les petits $\lambda$, qui récompensent l'apprentissage par cœur : ce découpage aurait même désigné $\lambda = 0{,}001$. La fuite fausse à la fois la mesure et le choix.

**e)** Le modèle final ($\lambda = 0{,}01$, standardisation et poids calculés sur les 30 groupes) a une perte de test de 0,356, mesurée une seule fois sur 10 groupes jamais vus. Elle est proche du score de validation croisée par groupe, 0,378, et loin du 0,246 des plis message par message. La validation croisée par groupe annonçait honnêtement ce qui attendait le filtre ; l'autre se trompait d'un tiers. L'écart restant entre 0,356 et 0,378 vient du hasard : 94 messages de test, c'est peu.

## Exercice 6.5 — Prédire, puis vérifier : la relance du théâtre

*Notions : écart en points (équation 6.5), erreur standard (équation 6.6), intervalle de confiance (équation 6.7).* Données D6.E5. La vidéo n'est utile que si elle fait gagner au moins 0,5 point.

### a) à e) Corrigé commenté

**a)** Taux A : $126 / 1\,800 = 7{,}0\ \%$. Taux B : $147 / 1\,750 = 8{,}4\ \%$. Écart : $8{,}4 - 7{,}0 = 1{,}4$ point de pourcentage. Variation relative : $1{,}4 / 7{,}0 = 20\ \%$.

**b)** En points : $7 \times 93 / 1\,800 \approx 0{,}362$ et $8{,}4 \times 91{,}6 / 1\,750 \approx 0{,}440$. $\text{ES} = \sqrt{0{,}362 + 0{,}440} = \sqrt{0{,}801} \approx 0{,}90$ point. Intervalle : $1{,}4 \pm 1{,}96 \times 0{,}90 \approx 1{,}4 \pm 1{,}75$, soit de **−0,35 à 3,15 points** environ.

**c)** *Le signe.* L'écart mesuré est positif, mais zéro est dans l'intervalle, et même de petits effets négatifs : l'essai ne permet pas d'affirmer que la vidéo a fait réserver davantage. *La précision.* L'intervalle mesure 3,5 points de large, deux fois et demie l'écart lui-même : l'effet est très mal connu. *L'intérêt pratique.* Le seuil de 0,5 point est dans l'intervalle, tout comme zéro et des effets de 3 points : la vidéo peut être très utile ou inutile, les données ne tranchent pas. Le collègue confond « l'intervalle contient zéro » et « l'effet est nul » : un essai trop petit ne prouve pas l'absence d'effet. 1,4 point reste la valeur la plus plausible ; ce qui manque, c'est la précision.

**d)** L'erreur standard varie comme $1 / \sqrt{n}$ : quatre fois plus de spectateurs divisent l'erreur standard, et donc la largeur de l'intervalle, par $\sqrt{4} = 2$, pas par 4.

**e)** $\text{ES} = \sqrt{0{,}801 / 4} \approx 0{,}45$ point, et l'intervalle va de $1{,}4 - 0{,}88 \approx 0{,}52$ à $1{,}4 + 0{,}88 \approx 2{,}28$ points. Zéro serait exclu, et le seuil de 0,5 point tout juste : si les taux restaient les mêmes, la vidéo serait jugée utile, de peu.

In [ ]:
def interval(conv_A, n_A, conv_B, n_B):
    """Renvoie (écart, borne basse, borne haute), en points de pourcentage."""
    pA, pB = conv_A / n_A, conv_B / n_B
    se = np.sqrt(pA * (1 - pA) / n_A + pB * (1 - pB) / n_B)   # équation 6.6
    return 100 * (pB - pA), 100 * (pB - pA - 1.96 * se), 100 * (pB - pA + 1.96 * se)
intervalle = interval   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de a), b) et e)
A, B = theater.iloc[0], theater.iloc[1]
r = interval(A['bookings'], A['audience'], B['bookings'], B['audience'])
if r is None:
    print('pas encore complété. Complétez la fonction.')
else:
    print(f'essai réel : écart {r[0]:.2f} points, intervalle de {r[1]:.2f} à {r[2]:.2f} →',
          'correct' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'à revoir')
    r4 = interval(4 * A['bookings'], 4 * A['audience'], 4 * B['bookings'], 4 * B['audience'])
    print(f'essai quatre fois plus grand : de {r4[1]:.2f} à {r4[2]:.2f} ; largeur {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'correct' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'à revoir')

In [ ]:
# f) 1 000 essais simulés dont le véritable effet vaut 1,4 point : combien d'intervalles contiennent 1,4 ?
rng = np.random.default_rng(65)
nA, nB, true_A, true_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, true_A, 1000) / nA
b = rng.binomial(nB, true_B, 1000) / nB
se_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contains = np.abs((b - a) - 0.014) <= 1.96 * se_sim
print(f'part des intervalles qui contiennent le véritable effet : {contains.mean():.1%}')
print(f'part des intervalles qui contiennent zéro : {np.mean(np.abs(b - a) <= 1.96 * se_sim):.1%}')

### f) Corrigé commenté

Environ 94 % des 1 000 intervalles simulés contiennent le véritable effet, 1,4 point : c'est le « 95 % », aux aléas de la simulation près. Il qualifie la **méthode** : sur un grand nombre d'essais, environ 95 % des intervalles ainsi construits contiennent le véritable effet. Remarquez aussi qu'avec un véritable effet de 1,4 point, une bonne partie des intervalles contiennent zéro : un essai de cette taille passe souvent à côté d'un effet réel. C'est une raison de plus de ne pas lire « contient zéro » comme « aucun effet ».

## Exercice 6.6 — Défi : combien de clients, et quand regarder ?

*Notions : erreur standard et intervalle de confiance (leçon 6.6), loi binomiale (leçon 4.5).*

### a) à c) Corrigé commenté

**a)** **Objectif de dimensionnement : la précision.** On veut que l'intervalle à 95 % ne s'écarte pas de plus de $m = 1$ point de l'écart mesuré. Ce n'est pas un calcul de puissance : on ne fixe ni hypothèse à tester ni probabilité de détecter un effet. **Hypothèses :** deux groupes tirés au sort de même taille $n$ ; taux supposés égaux à $p_A = 8$ % et $p_B = 10$ %, ceux de l'essai de la leçon 6.5 ; approximation par la courbe en cloche ($z = 1{,}96$) ; intervalle lu une seule fois, à la fin.

En points, $\text{ES} = \sqrt{\big(p_A(100 - p_A) + p_B(100 - p_B)\big) / n}$, et la condition $z \times \text{ES} \le m$ donne :

$$n \ge \frac{z^2 \, \big(p_A(100 - p_A) + p_B(100 - p_B)\big)}{m^2} = \frac{1{,}96^2 \times (8 \times 92 + 10 \times 90)}{1^2} = 3{,}8416 \times 1\,636 \approx 6\,284{,}9 .$$

On arrondit à l'entier supérieur : **6 285 clients par groupe**, environ 12 570 en tout. Ces 6 285 clients correspondent à une marge **approximative** d'un point : approximative parce que la courbe en cloche est une approximation, et parce que le calcul suppose des taux de 8 % et 10 %. Si les vrais taux diffèrent, la taille change : avec des taux plus proches de 50 %, il faudrait davantage de clients.

**b)** L'intervalle irait de $2{,}0 - 1{,}0 = 1{,}0$ à $3{,}0$ points : il toucherait le seuil sans l'exclure. Pour une demi-largeur de 0,5 point, il faut diviser l'erreur standard par 2, donc multiplier $n$ par 4 : environ 25 140 clients par groupe, soit 50 280 en tout. C'est plus que les 40 000 abonnés : une seule lettre n'y suffirait pas ; il faudrait par exemple répéter l'essai sur deux envois.

**c)** Non : on s'attend à beaucoup plus que 5 %, car chaque soir donne au hasard une nouvelle occasion de franchir la limite.

In [ ]:
def group_size(pA, pB, half_width):
    """pA, pB en pourcentage ; renvoie le plus petit n entier tel que 1,96 × ES ≤ half_width."""
    return int(np.ceil((1.96 / half_width) ** 2 * (pA * (100 - pA) + pB * (100 - pB))))
taille_par_groupe = group_size   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de a) et b)
n1 = group_size(8, 10, 1)
if n1 is None:
    print('pas encore complété. Complétez la fonction.')
else:
    print(f'demi-largeur 1 point : {n1} clients par groupe →', 'correct' if n1 == 6285 else 'à revoir')
    print(f'demi-largeur 0,5 point : {group_size(8, 10, 0.5)} clients par groupe, '
          f'{2 * group_size(8, 10, 0.5)} en tout')

In [ ]:
# d) 2 000 essais SANS effet. Calendrier : 250 clients par jour et par groupe, 20 lectures (une chaque soir,
#    sur les clients cumulés : 250, 500… 5 000 par groupe). Règle d'arrêt : premier soir où l'intervalle exclut 0.
rng = np.random.default_rng(66)
trials, days, per_day, p = 2000, 20, 250, 0.08
cA = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # commandes cumulées, groupe A
cB = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # groupe B : même taux, aucun effet
n = per_day * np.arange(1, days + 1)                                  # clients par groupe à la fin de chaque jour
pA_, pB_ = cA / n, cB / n
se_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
excludes = np.abs(pB_ - pA_) > 1.96 * se_j                                # l'intervalle du soir exclut-il zéro ?
end, one_evening = excludes[:, -1].mean(), excludes.any(axis=1).mean()
print(f'intervalle lu seulement le 20e jour : {end:.1%} des essais concluent à tort à un effet')
print(f'intervalle lu chaque soir, arrêt au premier « succès » : {one_evening:.1%}')

### d) et e) Corrigé commenté

Conditions de la simulation : 2 000 essais sans effet (taux de 8 % dans les deux groupes), 250 nouveaux clients par jour et par groupe, 20 lectures au plus, une chaque soir du 1^er^ au 20^e^ jour, sur les clients cumulés depuis le début (de 250 à 5 000 par groupe) ; règle d'arrêt : on arrête au premier soir où l'intervalle à 95 % exclut zéro, d'un côté ou de l'autre, et l'on conclut à un effet. Lu une seule fois, au 20^e^ soir, l'intervalle exclut zéro dans environ 5 % des essais sans effet (4,6 % dans la simulation) : c'est le risque prévu. Avec la lecture chaque soir et la règle d'arrêt, on conclut à tort dans environ un essai sur quatre (24,7 %). Ce chiffre dépend du calendrier : plus de lectures, ou des lectures plus rapprochées, le feraient encore monter. Chaque soir est une nouvelle occasion, pour le hasard, de faire sortir l'intervalle de zéro ; s'arrêter à ce moment-là, c'est garder les occasions favorables et jeter les autres. La règle : **fixer à l'avance la taille de l'essai et la date de lecture**, par exemple 6 300 clients par groupe d'après a), et lire l'intervalle à cette date. Regarder les chiffres en cours de route n'est pas interdit, mais ne doit pas décider de l'arrêt.